# Run Selected 10 Experiments With Groq

This notebook runs the selected-10 ablation suite against the Groq API with:

- local disk caching for every prompt
- shared Stage 1 and Stage 2 artifacts across configs
- resumable JSON outputs under `outputs/groq_selected10/`
- a low-hit shortcut where `targeted_attacks` reuses the same no-graph judgments as `six_agents`

Default topic source: `data/eval/google_form/form_topics.jsonl`

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "scripts").exists():
        ROOT = candidate
        break

sys.path.append(str((ROOT / "scripts").resolve()))

import groq_selected10_notebook as groq_nb

TOPIC_FILE = ROOT / "data" / "eval" / "google_form" / "form_topics.jsonl"
OUTPUT_ROOT = ROOT / "outputs" / "groq_selected10"
CACHE_DIR = OUTPUT_ROOT / "cache"
MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-120b")
FORCE = False
PAIR_BATCH_SIZE = 40

print("Topic file:", TOPIC_FILE)
print("Output root:", OUTPUT_ROOT)
print("Model:", MODEL)
print("GROQ_API_KEY set:", bool(os.environ.get("GROQ_API_KEY")))

Topic file: C:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\data\eval\google_form\form_topics.jsonl
Output root: C:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\outputs\groq_selected10
Model: openai/gpt-oss-120b
GROQ_API_KEY set: True


In [2]:
topics = groq_nb.load_selected_topics(TOPIC_FILE)
plan = groq_nb.estimate_request_plan(topics)
print(json.dumps(plan, indent=2))

{
  "topic_count": 10,
  "minimum_new_request_groups": {
    "baselines": 20,
    "stage1_2agent": 40,
    "stage1_6agent": 40,
    "stage4_unique_configs": 60
  },
  "notes": [
    "Stage1 is compressed to 4 requests per topic per agent-count setting.",
    "Stage2 is the dominant cost; pair batches are cached and reused.",
    "six_agents and targeted_attacks intentionally share no-graph judgments to save hits.",
    "full reuses six-agent targeted Stage1 and Stage2, then only adds Stage3 and graph-grounded Stage4."
  ]
}


In [3]:
result = groq_nb.run_selected10_suite(
    model=MODEL,
    topic_file=TOPIC_FILE,
    output_root=OUTPUT_ROOT,
    cache_dir=CACHE_DIR,
    force=FORCE,
    pair_batch_size=PAIR_BATCH_SIZE,
)
print(json.dumps(result, indent=2))

c:\Users\acer\OneDrive - Cloud Catalyst Asia\Personal\MAJ-Debate\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: 429: {"error":{"message":"Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kmqstnpkf2brwkpvm5pzhvtj` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197910, Requested 3398. Please try again in 9m25.056s. Need more tokens? Upgrade to Dev Tier today at https://

In [ ]:
summary_rows = result["summary_rows"]
try:
    import pandas as pd
    display(pd.DataFrame(summary_rows))
except Exception:
    print(json.dumps(summary_rows, indent=2))

In [ ]:
summary_path = Path(result["summary_json"])
print(summary_path)
print(summary_path.read_text(encoding="utf-8"))